# BERT Sentiment Analysis of Yelp Reviews

## Importing Dependencies

__Data manipulation & Visualization__:
- `pandas` and `numpy` libraries for numerical operations and easier data manipulation.
- `Seaborn` and `matplotlib` for visualization of training and evaluation results.
- `Counter` class from Python's `collections` module for counting occurences of labels and validating class distribution balance.

__Dataset Handling__:
- Hugging Face `datasets` library for efficient dataset handling and integration with transformer models.
- `RandomUnderSampler` from imbalanced-learn library for handling class imbalance through randomly undersampling the majority classes.

__Hugging Face__:
From the transformers library, import the following classes and functions:
- `Trainer` for training and evaluating the model performance.
- `TrainingArguments` class for specifying training hyperparameters and configurations (i.e., number of epochs, learning rate, etc.).
- `AutoTokenizer` class for loading the appropriate tokenizer based on the selected model and tokenization of text data.
- `AutoModelForSequenceClassification` class to load a pre-trained BERT model for sequence classification (in this case, sentiment analysis).

__LoRA (Low-Rank Adaptation)__:
- Importing LoRA configuration and the `get_peft_model` wrapper from the parameter-efficient fine-tuning library.

__Evaluation Metrics__:
- Improting accuracy score, precision, recall, f1-score, and confusion matrix from `sklearn.metrics` for final model evaluation.

In [ ]:
# Data manipulation & visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Dataset handling
from datasets import Dataset
from imblearn.under_sampling import RandomUnderSampler

# Hugging Face
from transformers import(
    Trainer,
    TrainingArguments,
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

# LoRA
from peft import LoraConfig, get_peft_model

# Evaluation
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# Ignore warnings
import warnings
warnings.filterwarnings("ignore", message=".*HF Hub.*")
warnings.filterwarnings("ignore", message=".*pin_memory.*")

## Data Preparation

### Loading the Previously Saved Dataset

- `reviews_df = pd.read_csv()`: Loading the previously cleaned and saved CSV file into a pandas DataFrame, assigning it to `reviews_df` variable.
- Print the number of rows loaded from the CSV file to verify that the dataset has been loaded correctly.
- `reviews_df.info()`: Display the basic info about the DataFrame (column names, non-null counts, and data types).

In [ ]:
# Load the cleaned CSV
reviews_df = pd.read_csv('../data/processed/yelp_reviews_cleaned.csv')
print(f"Loaded {len(reviews_df)} rows from CSV\n")
print(reviews_df.info())

### Label Mapping

Map the three sentiment labels (positive, negative and neutral) to the corresponding star ratings in the DataFrame for training. BERT labels are 0, 1, 2 for negative, neutral and positive respectively.

- `id2label`: A dictionary mapping label indices to their corresponding string labels (e.g., 0 to "NEGATIVE", 1 to "NEUTRAL", and 2 to "POSITIVE"). Later will be used to convert model predictions back to human-readable labels.
- `label2id`: A dictionary mapping the string labels ("NEGATIVE", "NEUTRAL", "POSITIVE") to their corresponding integer labels (0, 1, 2) for model training.

In [ ]:
reviews_df['label'] = reviews_df['stars'].replace({
    1: 0, # negative
    2: 0, 
    3: 1, # neutral
    4: 2,
    5: 2  # positive
})

id2label = {0: "NEGATIVE", 1: "NEUTRAL", 2: "POSITIVE"} # A map from index to label
label2id = {"NEGATIVE": 0, "NEUTRAL": 1, "POSITIVE": 2} # A map from label to index

### Checking for Class Imbalance

Display the distribution of labels in the dataset to check for class imbalance:
- `reviews_df['label'].value_counts()`: Counts the occurrences of each unique value in the 'label' column of the `reviews_df` DataFrame.
- `.sort_index()`: Sorts the value counts by the label index (0, 1, 2) for readability.

In [ ]:
# Check for class imbalance in the dataset
print(f"Distribution of labels in the dataset:\n{reviews_df['label'].value_counts().sort_index()}")

### Balancing the Dataset

In this code block, an undersampling strategy is applied to balance the dataset through reducing the number of samples form the majority classes (negative and positive) to match the number of samples in the minority class (neutral). To do this, the `RandomUnderSampler` class from the imbalanced-learn library is employed. It resamples the dataset by randomly removing samples from th majority classes until they match the number of samples in the minority class.

__Defining the data and target features__:
To prepare for undersampling, the dataset is split into features (inputs) and targets (outputs/labels). The features (X) are the cleaned review texts, and the targets (y) are the corresponding sentiment labels.
- `X = reviews_df[['cleaned_text']]`: Converting the input series to a DataFrame with double brackets as `RandomUnderSampler` expects 2D features.
- `y = reviews_df['label']`: Assign the 'label' column of the DataFrame to `y` (the target variable for undersampling).

__Creating the undersampler object__:
Creating an instance of the `randomUnderSampler` class and specifying the sampling strategy. Parameters:
- `sampling_strategy='not minority'`: Resample all classes but the minority class (e.g, downsampling the majority classes to match the minority class count).
- `random_state=42`: Sets the seed for reproducibility, meaning that the same samples are selected for undersampling each time the code is run.

__Fitting and Transforming the Data__:
Employing the `fit_resample` method to perform undersampling on the dataset, storing the resulting balanced features and labels in `X_res` and `y_res` variables respectively. The `fit_resample` method consists of two steps: `fit` analyzes which samples to remove based on the specified srategy, and `resample` returns the new balanced dataset after undersampling.

__Creating a Balanced DataFrame__:
Storing the transformed features and labels in a new DataFrame and assigning it to the `balanced_reviews_df` variable. Mapping the resampled variables to 'cleaned_text' and 'label' columns in a new DataFrame:
- `'cleaned_text': X_res['cleaned_text']`: Unpacking the transformed 'cleaned_text' column from 2D DataFrame to 1D Series after undersampling.
- `'label': y_res`: Assign the resampled labels to the 'label' column in the balanced DataFrame.

__Adding Label Mappings__:
Mapping the numeric sentiment labels back to strings using the `id2label` dictionary for better readability, and assigning them to a new `'label_names'` column in the balanced DataFrame.

In [ ]:
# Define the data and target features for undersampling
X = reviews_df[['cleaned_text']]
y = reviews_df['label']

undersampler = RandomUnderSampler(sampling_strategy='not minority', random_state=42) # Controlling randomness by setting a seed for reproducibility -- the magical 42 ("Answer to the Ultimate Question of Life")

X_res, y_res = undersampler.fit_resample(X, y)

# Verify the new distribution of labels after undersampling
print(f"Before undersampling: {Counter(y)}")
print(f"After undersampling: {Counter(y_res)}")

balanced_reviews_df = pd.DataFrame({
    'cleaned_text': X_res['cleaned_text'],
    'label': y_res 
})

balanced_reviews_df['label_names'] = balanced_reviews_df['label'].replace(id2label)

print(f"\nDisplay the first few rows of the balanced dataset:\n{balanced_reviews_df.sort_index().head(15)}")

### Converting the DataFrame to a Dataset Object

- Converting the balanced DataFrame into a Hugging Face Dataset object using the `.from_pandas()` method, assigning it to `balanced_reviews_hf` for further processing and model training. 
- Saving the balanced dataset to disk using the Hugging Face `save_to_disk` method for future use.

In [ ]:
# Convert the balanced DataFrame to a Hugging Face Dataset object
balanced_reviews_hf = Dataset.from_pandas(balanced_reviews_df)
print(balanced_reviews_hf[:10])

# Save the HuggingFace Dataset to disk
balanced_reviews_hf.save_to_disk('../data/processed/yelp_reviews_balanced_dataset')

## Loading the Model

Loading the pre-trained bert-base-uncased model from the Hugging Face Hub with the `.from_pretrained()` method, specifying the parameters for sequence classification and the label mappings.
- `AutoModelForSequenceClassification`: A class form Hugging Face Transformers library, allowing for loading and fine-tuning language models specifically for sequence classification tasks, such as sentiment analysis.
- `.from_pretrained`: loads the pre-trained base BERT model. 

__Parameters__:
- `num_lables=3`: Defining three sentiment classes (negative, positive, neutral).
- `id2label=id2label`: A mapping form numerical labels to string labels.
- `label2id=label2id`: A mapping form string labels to numerical labels.

In [ ]:
# Load the pre-trained BERT model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained('google-bert/bert-base-uncased', num_labels=3, id2label=id2label, label2id=label2id)

## Creating a TextTokenizer Class

Creating a `TextTokenizer` class for loading the BERT tokenizer and tokenizing the text data in the Hugging Face Dataset.

__1. The constructor Method__:
The `__init__` method is called automatically when an instance of a class is created. It initializes the object's state and its attributes/starting values.

__Parameters__:
- `self`: Represents the instance of the class itsef. It references the current object and lets it access its attributes and methods.
- `model_name="google-bert/bert-base-uncased"`: Define the default model parameter. If no model name is provided when creating an instance of this class, it will default to "google-bert/bert-base-uncased" model.

__Function Body__:
Loading the tokenizer for the bert-base-uncased model using the `.from_pretrained()` method from the Hugging Face `AutoTokenizer` class. Assigning the loaded tokenizer to an instance variable `self.tokenizer`.



__2. The `tokenize` Method__:
The `tokenize` method takes a batch of texts from the `'cleaned_text'` column of the DataFrame, verifies whether the inputs are strings, and applies the loaded tokenizer to convert the raw text into token IDs (`'input_ids'`) and attention masks (`'attention_mask'`).

__Parameters__:
- `text_batch`: The input batch of text data to be tokenized.
- `max_length=128`: Maximum sequence length for tokenization set to default 128 tokens.

__Function Body__:
- Ensuring each input is a string with a list comprehension, which loops over each text in the `'cleaned_text'` column of the batch and converts it to a string with a `str()` call, creating a new list of strings to pass into the tokenizer.
- Calling the tokenizer on the list of strings (the `texts` variable), specifying the parameters for processing the text data.
- `max_length=max_length`: Sets the maximum sequence length to default.
- `padding="max_length"`: Ensures equal length by padding shorter sequences to `max_length`.
- `truncation=True`: Truncates longer sequences to match the maximum length (`max_length`).



__3. The `decode` Method__:
Defining the `decode` method to convert the transformed tokens (`'input_ids'`) back into human-readable format (e.g., back to `'cleaned_text'`).

__Parameters__:
- `token_ids_batch`: A batch of token IDs (`'input_ids'`) to decode.
- `skip_special_tokens=True`: Removes special tokens like [CLS], [SEP], and [PAD] from the decoded output for better readability.

__Function Body__:
Returns a list of decoded strings by calling a `.batch_decode()` method on the on the BertTokenizer instance.

In [ ]:
class TextTokenizer:
    def __init__(self, model_name="google-bert/bert-base-uncased"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    def tokenize(self, text_batch, max_length=128):
        texts = [str(text) for text in text_batch["cleaned_text"]]
        return self.tokenizer(texts,
                              max_length=max_length,
                              padding="max_length",
                              truncation=True
                              )
    
    def decode(self, token_ids_batch, skip_special_tokens=True):
        return self.tokenizer.batch_decode(token_ids_batch, skip_special_tokens=skip_special_tokens)

### Applying the Tokenization Function

- Creating an instance (object) of the class by calling the constructor method `__init__()` to initialize the tokenizer for "google-bert/bert-base-uncased" (default), and assigning it to the `'tokenizer_obj'` variable.
- Using a Hugging Face Dataset's `.map()` method to apply the `'tokenize'` function to the earlier balanced dataset. The `batched=True` parameter allows for processing and applying the function to multiple samples in a batch, which is more efficient than processing samples one by one.

In [ ]:
tokenizer_obj = TextTokenizer() # loads the tokenizer for "google-bert/bert-base-uncased" (default)

tokenized_dataset = balanced_reviews_hf.map(tokenizer_obj.tokenize, batched=True)

### Inspecting the Tokenized Dataset

- Printing the keys of the tokenized dataset by accessing the `'column_names'` attribute of the dataset to verify the presence of `'input_ids'`, `'attention_mask'`, and `'label'` columns after tokenization.
- Printing the first 2 tokenized `input_ids` for verification that the tokenization worked correctly.
- Printing the first 2 decoded sentences to verify that the `'decode'` method works correctly on the tokenized `'input_ids'`.

In [ ]:
# Display the keys of the tokenized dataset to verify the presence of 'input_ids', 'attention_mask', and 'label'
print(f"Keys in the tokenized dataset: {tokenized_dataset.column_names}\n")

# Display the tokenized sentences to verify the tokenization process
print(f"First 2 tokenized input_ids:\n{tokenized_dataset['input_ids'][:2]}\n")

# Display the first 2 decoded sequences from the tokenized dataset
print(f"First 2 decoded sequences:\n{tokenizer_obj.decode(tokenized_dataset['input_ids'][:2])}")

## Splitting the Dataset

In this code block, the `'train_test_split'` function from the scikit-learn library is used to divide the earlier balanced and tokenized dataset into two subsets: a training set (80% of the data) and a validation set (the reamining 20% of the data). This function randomly selects samples for the training and validation sets and ensures that the distribution of lables is balanced across both subsets. The `train_test_split` method automatically shuffles the data by default (`shuffle=True`).

__Parameters__:
- `test_size=0.2`: Define the size of the test set (`train_size` = 1 - `test_size`).
- `seed=42`: Set seed for reproducibility of the train/test split.

__Splitting into sets__:
- Creating two separate variables (`'train_set'` and `'test_set'`) after splitting to store the training and validation datasets for easier access during model training and evaluation.
- Verifying the sizes of the train and validation sets after splitting by calling the `len()` function on both sets.

In [ ]:
# Split 80% train / 20% validation
dataset_split = tokenized_dataset.train_test_split(test_size=0.2, seed=42)

train_set = dataset_split['train']
test_set = dataset_split['test']

print(f"Train size: {len(train_set)}, Validation size: {len(test_set)}")

### Dataset Format Conversion

Converting the Hugging Face Dataset Objects (`'train_set'` and `'test_set'`) into PyTorch tensors (multi-dimensional arrays) for compatibility with BERT and training efficiency. This preprocessing step is done using a `'.set_format'` method from the Hugging Face Datasets library, which allows for conversion of selected columns in the dataset into a specified format (in this case, PyTorch tensors).

__Parameters__:
- `type='torch'`: Specifying the format for dataset conversion to PyTorch tensors.
- `columns=['input_ids', 'attention_mask', 'label']`: Defining which columns should be converted to the specified format.

In [ ]:
train_set.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_set.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

## LoRA Configuration

Initializing the LoRA configuration through the `LoraConfig` class, which allows for hyperparameter specification.

__Parameters__:
- `r=16`: The rank, determines the number of trainable parameters.
- `lora_alpha=32`: The scaling factor set to be twice the rank, controls the magnitude of the updates during training.
- `target_modules=['query', 'value']`: Target only the query and value tensors as they have the strongest impact on model behaviour.
- `lora_dropout=0.1`: Regularization by adding small random stochastic noise to prevent overfitting and improve generalization.
- `bias='none'`: Do not train the bias terms in the model, only update the query and value tensors (as specified in `target_modules`).
- `task_type='SEQ_CLS'`: Define the task type as sequence classification for sentiment analysis.

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['query', 'value'],
    lora_dropout=0.1,
    bias='none',
    task_type='SEQ_CLS'
)

### Applying LoRA to the Pre-Trained BERT Model

__Merging the LoRA model with the base BERT model__:
- `get_peft_model(model, lora_config)`: Wrapping the original BERT model with `get_peft_model` to apply the LoRA configuration for fine-tuning. This allows to only fine-tune the injected low-rank matricies, while keeping the original model parameters frozen (meaning, the gradients will not be propagated through those paramerters during training).
- `lora_model`: Assign the merged LoRA model to a new variable.
- `lora_model.print_trainable_parameters()`: Displaying the number of trainable parameters in the LoRA model.

__Iterating over the named parameters__:
- `for name, param in lora_model.named_parameters()`: A for loop to iterate through the model's named parameters. The `name, param` tuple refers to the name of the parameter and the parameter itself.
- `print(name, param.requires_grad)`: Display the name of each parameter and whether it requires gradient updates (i.e., whether it's trainable or frozen).

In [ ]:
lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters() 

for name, param in lora_model.named_parameters():
    print(name, param.requires_grad)

## Training Configuration (`TrainingArguments`)

In this block, a `training_args` object is created to specify the training parameters for the Trainer.

- `output_dir="./lora_results"`: Automatically creates a "lora_results" directory to save model checkpoints and logs.
- `per_device_train_batch_size=8`: Defines the number of training sample batches processed on a GPU before the model's internal parameters are updated. Set to 8 to balance training speed and memory usage.
- `per_device_eval_batch_size=8`: Analogous for evaluation batch size, defines the number of evaluation sample batches processed on a GPU.
- `learning_rate=2e-4`: A relatively high learning rate for LoRA fine-tuning, as only a small number of parameters (low-rank matrices) are being updated, allowing for faster convergence.
- `num_train_epochs=3`:` A standard number of epochs for fine-tuning, means there will be three full passes over the dataset, allowing the model to update its weights at the end of each epoch. Set to <10 to avoid overfitting, can be adjusted based on validation performance.
- `eval_strategy="epoch"`: Evaluate the model at the end of each epoch to monitor performance.
- `save_strategy="epoch"`: Save the model updates at the and e=of each epoch.
- `logging_strategy="steps"`: Log training metrics every X steps (defined by logging_steps) instead of the end of every epoch for more frequent insights into the training process.
- `logging_steps=50`: Define the number of steps between logging events.
- `report_to='none'`: Disable integration with external logging platforms like WandB or TensorBoard, local logging is sufficient for small projects.

In [ ]:
training_args = TrainingArguments(
    output_dir="./lora_results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8, 
    learning_rate=2e-4,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    report_to="none"
)

## Defining a Function for Computing Evaluation Metrics

Defining a function for computing evaluation metrics during model training. It will be passed to the Trainer's `compute_metrics` argument to automatically calculate metrics for model's predictions. 

__Input__:
- `eval_prediction`: A tuple storing the model's logits and the true labels. The Trainer passes this as a single argument to the `compute_metrics` callback.
- `logits`: The raw model output scores before applying softmax probability conversion.
- `labels`: The true class labels for the evaluation dataset.

__Converting Logits into Predictions__:
Converting model logits into predictions by taking the index of the highest logit value for each example with `np.argmax()`.
- `np.argmax()`: Finds the index of the maximum value along the specified axis.
- `axis=-1`: Refers to the last dimension of the array.


__Accuracy__:
- `accuracy_score()`: A built in function from scikit-learn. Calculates the accuracy of the predictions through a ratio of correct predictions to total samples.
- Compares the predicted labels (`predictions`) with the true labels (`labels`).


__Precision, Recall, F-Score__:
- `precision_recall_fscore_support()`: A built-in function from scikit-learn, returns a tuple of `(precision, recall, f1, support)` for the specified average method.
- `average='weighted'`: Computes evaluation metrics for each class separately and returns a weighted average based on the number of true instances (support) in that class.
- `support(_)`: Support is the number of true positive instances for each class. Ignored with a `'_'` because it is not needed for the weighted average calculation.

In [ ]:
def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
 
    predictions = np.argmax(logits, axis=-1) 

    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted') # Each class's metrics are multiplied by its support and averaged over all classes

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

## Training the Model

In this code block, the `Trainer` class from the Hugging Face Transformers library is used to simplify the training loop and automate the model fine-tuning process, allowing for customization of training parameters. 
The `trainer` object is created, which facilitates the training process and model performance evaluation using the methods provided by the `Trainer` class.

__Parameters__:
- `model=lora_model`: Specifying the LoRA-wrapped BERT model as the model to train.
- `args=training_args`: Passing the earlier created `training_args` object which contains all hyperparameters for training (e.g., batch size, learning rate, epochs, etc.).
- `train_dataset=train_set`: The training subset of the tokenized dataset converted into PyTorch format. This is the data used for gradient updates.
- `eval_dataset=test_set`: The validation/test subset of the tokenized dataset converted into PyTorch format. This will be used to monitor model performance and compute evaluation metrics (accuracy, precision, F-score, etc.).
- `compute_metrics=compute_metrics`: Passing the earlier defined function for evaluation metrics computation, which will be called after each training epoch. 

__Calling the `.train()` Method__:
Calling the `.train()` method of the class on the `trainer` object. This method starts the training loop, which consists of 3 epochs of running over the entire training dataset. During the training process, the sample batches from the `train_set` are fed into the model with a forward pass, the model logits and cross-entropy loss are computed. During the backward pass, the gradients are computed and only propagated back through the LoRA parameters, while all other model parameters remain frozen.

In [ ]:
trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=train_set, 
    eval_dataset=test_set,
    compute_metrics=compute_metrics
)

trainer.train()

### Running Inference on the Test Dataset 
In this code block, the `.predict()` method of the Trainer class is used to run inference on the `test_set`, generate predictions, and extract the prediction results for further analysis. This method returns a named tuple `PredictionOutput`, which contains the predictions, labels, and computed metrics.

- `trainer.predict(test_set)`: Performs inference (prediction) on the test dataset by loading the earlier trained model (in this case, the LoRA-fine-tuned BERT). It batches the samples from the dataset and runs the model on each batch to generate logits (raw model outputs).
- `prediction_output`: A variable storing the `PredictionOutput` named tuple for later attribute access.
- `prediction_output.metrics`: A dictionary containing the computed metrics (e.g., `'test_accuracy'`, `'test_precision'`, `'test_recall'`, etc.).

In [ ]:
prediction_output = trainer.predict(test_set)
print(f"Prediction Metrics: {prediction_output.metrics}") # Display the dictionary of evaluation metrics 

- Accessing the `.predictions` attribute/property of the `prediction_output` object. This is a NumPy array containing raw model predictions (logits) before softmax application, assigned to a corresponding variable.
- Accessing the `.label_ids` attribute of the `prediction_output` object, which is a NumPy array of the true labels from the dataset (e.g., the targets). ssigned to the `labels` variable for later comparison with predictions.

In [ ]:
logits = prediction_output.predictions # Raw model predictions (logits) before softmax
labels = prediction_output.label_ids # True labels from the test set

### Converting Logits to Class Predictions

Here, the `argmax` function is employed to convert the raw logits into discrete class predictions. In classification tasks, the class with the highest logit value is considered to be the model's prediction. In this context, the `argmax` function is sufficient as it yields the same class prediction results as `softmax`.

- `np.argmax(logits, axis=1)`: Selects the index of the highest logit value for each sample.
- `axis=1`: Finds the indices of the highest logits along each row (i.e., across all class score in a single example).

In [ ]:
predictions = np.argmax(logits, axis=1)

In [ ]:
# Display first 10 predictions and their corresponding true labels for verification
print(f"First 10 predictions: {predictions[:10]}")
print(f"First 10 true labels: {labels[:10]}")

## Prediction Analysis

__Mapping Sentiment Labels Onto Prediction Results__:
For improved readability and interpretability of model's outputs, human-readable sentiment labels were mapped to both the predicted and true class labels using list comprehentions. A label mapping from the `id2label` dictionary was applied to each numeric class prediction ID by iterating through the `predictions` and `labels` arrays. The resulting mappings were assigned to the `prediction_labels` and `true_labels` variables.

__Displaying the Prediction Results__:
Creating a loop that iterates over the first 10 samples in the test dataset, printing the decoded review texts along with the corresponding true and predicted class labels for clean comparison.

__Token IDs Decoding__:
- `tokenizer_obj.decode(...)`: Converting token IDs (`'input_ids'`) back to human-readable string format.
- `test_set[i]['input_ids']`: Accessing the `'input_ids'` tensor from the i-th sample in the test dataset.

__Print Class Labels__:
- True Label: `true_labels[i]`
- Predicted Label: `prediction_labels[i]`

In [ ]:
prediction_labels = [id2label[prediction] for prediction in predictions]
true_labels = [id2label[label] for label in labels]

for i in range(10):
    print(f"Review: \n{tokenizer_obj.decode(test_set[i]['input_ids'])}")
    print(f"True Label: {true_labels[i]}, Predicted Label: {prediction_labels[i]}\n")

### Creating a Confusion Matrix

In this section, a confusion matrix is created to evaluate the overall performance of the sentiment classification model. This is done by implementing a `confusion_matrix` function from the scikit-learn library, which compares the predicted lables(`predictions`) with the true class labels (`labels`). 

__Interpretation__:
- The diagonal values (from top-left to bottom-right) represent the correctly classified instances.
- The rows represent the actual (true) class labels.
- The columns represent the predicted class labels.
- The off-diagonal values are class misclassifications.

In [ ]:
conf_matrix = confusion_matrix(labels, predictions)
print(f"Confusion Matrix:\n{conf_matrix}")

## Creating a Seaborn Heatmap for Confusion Matrix Visualization

In this section, a heatmap is generated to provide a visual representation of the confusion matrix using the `Seaborn` library.

__Parameters__:
- `confusion_matrix`: The data parameter, a 2D array of counts.
- `annot=True`: Displays the numerical values in each cell of the heatmap.
- `fmt='d'`: Formats the annotations as integers.
- `cmap='coolwarm'`: Sets the color scheme of the heatmap.
- `xticklabels=id2label.values()`: Custom labels for the x-axis (predicted sentiment classes).
- `yticklabels=id2label.values()`: Custom labels for the y-axis (true sentiment classes).

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='coolwarm', xticklabels=id2label.values(), yticklabels=id2label.values())